# Practice: Match Your Question to a Method

**Computational Text Analysis Pack · Module 01 — Framing the Question**

*Technique:* Turning a humanities question into a countable one, or deciding it is not

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/00-foundations/01c_practice_match_question.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **You have a research question in ordinary language. Which of this course's
> methods, if any, can answer it — and what would you have to give up to make it
> countable?**

Moretti calls this *operationalising*: building a bridge from a concept to a
measurement. The bridge always costs something, and naming the cost is the
scholarly part.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# Four real questions, worked from concept to measurement.
CASES = [
 ("Did colonial writers dehumanise Africans?",
  "count words from a dehumanising lexicon per 1,000 words, by author",
  "M15d dictionary analysis + M4 keyness",
  "LOST: irony, context, and who counts as dehumanising. The dictionary IS the argument."),
 ("Is this anonymous pamphlet by X?",
  "Cosine Delta over 2,000 MFW, with leave-one-out accuracy on known works",
  "M9d + M9g",
  "LOST: nothing much, IF the candidate set is right. Everything, if the true author is not in it."),
 ("How did coverage of health change after 2020?",
  "not answerable",
  "none",
  "This corpus has no dates. No method fixes a missing column."),
 ("Which stories travelled between outlets?",
  "shingle overlap, cluster, then read the aligned pairs",
  "M18a-d",
  "LOST: direction. Similarity is undirected; priority needs dates or philology."),
]
for q, measure, modules, cost in CASES:
    print(f"Q: {q}")
    print(f"   measure : {measure}")
    print(f"   modules : {modules}")
    print(f"   {cost}\n")

In [ ]:
# Work case 1 far enough to see the cost.
LEXICON = {"savage", "savages", "primitive", "barbarous", "heathen", "native",
           "natives", "tribe", "tribes", "uncivilised", "uncivilized"}
works = (df_colonial.groupby(["work_id", "author"])["text"]
         .apply(" ".join).reset_index())
rows = []
for _, r in works.iterrows():
    toks = tokenise(r["text"])
    if len(toks) < 3000: continue
    hits = sum(1 for w in toks if w in LEXICON)
    rows.append({"author": r["author"], "rate_per_1000": hits/len(toks)*1000, "hits": hits})
res = pd.DataFrame(rows).groupby("author").agg(
    works=("hits", "size"), mean_rate=("rate_per_1000", "mean")).sort_values(
    "mean_rate", ascending=False)
print(res.head(10).to_string(float_format=lambda v: f"{v:.2f}"))

In [ ]:
# Now the cost, made concrete: read the contexts before believing the ranking.
top_author = res.index[0]
text = " ".join(works.loc[works["author"] == top_author, "text"])
toks = tokenise(text)
shown = 0
print(f"contexts for lexicon hits in {top_author}:\n")
for i, w in enumerate(toks):
    if w in LEXICON and shown < 6:
        print("   ..." + " ".join(toks[max(0,i-10):i+11]) + "...")
        shown += 1
print()
print("Check each line. Is the word used by the narrator, quoted from someone")
print("else, or discussed critically? A dictionary count cannot tell the")
print("difference, and in a corpus of writing ABOUT colonialism that difference")
print("is often the whole point.")

## The operationalisation worksheet

For your own question, fill these in before writing any code.

1. **The concept.** State it in one sentence, in ordinary language.
2. **The observable.** What in the text would change if the concept were more or
   less present?
3. **The measurement.** How would you count that observable?
4. **The unit.** Per document, per author, per 1,000 words?
5. **The comparison.** More than *what*? A measurement without a reference is
   not a finding.
6. **The cost.** What does the measurement miss that the concept includes?

If step 2 is hard, the question may not be countable — and saying so is a
legitimate outcome. Case 3 above is not a failure; it is a corpus requirement
discovered before the work rather than after.

### 🔧 Your turn 1

Fill in the worksheet for your own question, then show step 6 to someone in your field. Do they accept the cost?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Step 6 is where most disagreements in computational humanities actually live —
not about the statistics, but about whether the operationalisation preserved
enough of the concept to be worth doing. That is a disciplinary judgement, not a
technical one, and it is the reason the Da–CLS exchange in module 11c never
resolved: the two sides were arguing about step 6 while appearing to argue about
step 3.

</details>